# Module 5.2 — Mean Reversion Strategies
## When Prices Remember Where They Came From

---

### On Stationarity and Spreads

**Momentum** bets on persistence; **mean reversion** bets on **transience**. At short horizons, microstructure and liquidity push prices around a fair value; at longer horizons, pairs bound by economic links (same sector, supply chain, index arbitrage) may **co-move** so that a **spread** is more stationary than either leg alone. **Statistical arbitrage** is the disciplined exploitation of these temporary dislocations—always under **model risk**, **estimation error**, and the possibility that the relationship **breaks** (regime change, corporate action, structural shift).

This notebook connects the standard toolkit: **cointegration** and **pairs trading**, **z-scores** and **Bollinger Bands**, the **Ornstein–Uhlenbeck (OU)** idealization and **half-life**, simple **multi-pair** portfolios, and **dynamic sizing** tied to signal strength and spread volatility.

---

### Learning Objectives

1. **Explain** statistical arbitrage as market-neutral spread trading with defined exit rules
2. **Simulate** and **estimate** OU-type mean reversion and **half-life**
3. **Test** cointegration (Engle–Granger) and build a **hedge ratio** from regression
4. **Backtest** a **pairs trading** rule on z-scored spreads with transaction costs
5. **Implement** **Bollinger Band** mean reversion on price
6. **Combine** multiple mean-reverting spreads with **risk-aware weights**
7. **Scale** positions dynamically with **signal strength** and spread volatility

---

### Module Contents

1. **Statistical arbitrage & spreads** — Economic idea and implementation risks
2. **Ornstein–Uhlenbeck & half-life** — Continuous-time intuition, discrete estimation
3. **Cointegration & pairs trading** — Hedge ratio, ADF on spread, trading algorithm
4. **Bollinger Bands & z-scores** — Band touch rules vs. standardized spread
5. **Multiple pairs** — Small portfolio of spreads
6. **Dynamic position sizing** — Vol targeting and z-scaled exposure

---

*"Reversion to the mean is a powerful force—until the mean moves."*


In [ ]:
# ========================================
# Setup & synthetic cointegrated pairs
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.regression.linear_model import OLS

from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams.update({"figure.figsize": (14, 6), "axes.titlesize": 15, "figure.dpi": 100})

COLORS = {
    "a": "#1565C0",
    "b": "#C62828",
    "spread": "#6A1B9A",
    "z": "#00838F",
    "pnl": "#2E7D32",
    "neutral": "#455A64",
}

print("=" * 78)
print(" " * 12 + "MODULE 5.2: MEAN REVERSION STRATEGIES")
print("=" * 78)
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


def simulate_ou_spread(n: int, theta: float, mu: float, sigma: float, s0: float, rng) -> np.ndarray:
    """Euler OU: dS = theta*(mu-S)*dt + sigma*dW, daily dt=1/252."""
    dt = 1 / 252
    s = np.zeros(n)
    s[0] = s0
    for t in range(1, n):
        s[t] = s[t - 1] + theta * (mu - s[t - 1]) * dt + sigma * np.sqrt(dt) * rng.standard_normal()
    return s


def simulate_cointegrated_pair(
    n_days: int = 1200,
    beta: float = 1.0,
    theta: float = 4.0,
    sigma_ou: float = 0.25,
    sigma_rw: float = 0.012,
    seed: int = 42,
) -> pd.DataFrame:
    """log P1 random walk; spread OU; log P2 = beta * log P1 + spread (cointegrated)."""
    rng = np.random.default_rng(seed)
    dt = 1 / 252
    log_p1 = np.zeros(n_days)
    for t in range(1, n_days):
        log_p1[t] = log_p1[t - 1] + sigma_rw * np.sqrt(dt) * rng.standard_normal()
    spread = simulate_ou_spread(n_days, theta=theta, mu=0.0, sigma=sigma_ou, s0=0.0, rng=rng)
    log_p2 = beta * log_p1 + spread
    p1, p2 = np.exp(log_p1) * 100, np.exp(log_p2) * 100
    idx = pd.bdate_range("2016-01-04", periods=n_days)
    return pd.DataFrame({"P1": p1, "P2": p2, "log_P1": log_p1, "log_P2": log_p2, "spread_true": spread}, index=idx)


pair_df = simulate_cointegrated_pair()
print(f"Pair sample: {len(pair_df)} days")
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(pair_df.index, pair_df["P1"], label="P1", color=COLORS["a"], lw=1)
ax[0].plot(pair_df.index, pair_df["P2"], label="P2", color=COLORS["b"], lw=1)
ax[0].legend()
ax[0].set_title("Synthetic cointegrated pair (prices)")
ax[1].plot(pair_df.index, pair_df["spread_true"], color=COLORS["spread"], lw=1)
ax[1].set_title("Latent mean-reverting spread (OU)")
ax[1].axhline(0, color="gray", lw=0.8)
plt.tight_layout()
plt.show()


---

## 1. Statistical Arbitrage & Spread Trading

### Why quants need this

**Statistical arbitrage** funds harvest small, frequent edges from **relative value**: long one asset, short another in proportions that make the **net exposure** approximately **market- and factor-neutral**. The PnL driver is the **mean reversion of a spread**, not directional beta. Quants need this framework because it is the backbone of **equity stat-arb**, **index arbitrage**, and many **pairs** desks—along with the hard lessons: **leverage**, **short borrow**, **margin**, and **relationship breaks** can turn a stationary model into a trend.

### Working spread

With hedge ratio $\beta$, a common log-price spread is
$$S_t = \log P^{(2)}_t - \beta \log P^{(1)}_t.$$
Trading rules often use the **z-score** $z_t = (S_t - \bar{S}_{t,n})/\hat{\sigma}_{t,n}$ over a rolling window $n$.


---

## 2. Ornstein–Uhlenbeck Process & Half-Life

### Why quants need this

The **OU process** is the canonical **mean-reverting** diffusion:
$$dX_t = \theta(\mu - X_t)\,dt + \sigma\,dW_t.$$
The **half-life** (time for expected deviation from $\mu$ to halve) is
$$t_{1/2} = \frac{\ln 2}{\theta}.$$
Quants use half-life to set **holding period expectations**, **rolling window lengths**, and to sanity-check whether a spread **reverts fast enough** to overcome costs.

### Discrete proxy

With $\Delta t = 1$, an AR(1) $X_{t+1} = c + \phi X_t + \varepsilon_t$ relates approximately $\phi \approx e^{-\theta \Delta t}$, hence $\theta \approx -\ln(\phi)/\Delta t$ and $t_{1/2} \approx \ln(2)/\theta$.


In [ ]:
def estimate_half_life(series: pd.Series) -> Dict[str, float]:
    """AR(1) on levels; return phi, theta (daily), half_life days."""
    x = series.dropna()
    y = x.shift(-1).dropna()
    x = x.loc[y.index]
    if len(x) < 50:
        return {"phi": np.nan, "theta": np.nan, "half_life_days": np.nan}
    X = sm.add_constant(x.values)
    model = OLS(y.values, X).fit()
    phi = float(model.params[1])
    if phi <= 0 or phi >= 1:
        return {"phi": phi, "theta": np.nan, "half_life_days": np.nan}
    theta = -np.log(phi)
    hl = np.log(2) / theta
    return {"phi": phi, "theta": theta, "half_life_days": hl}


hl_stats = estimate_half_life(pd.Series(pair_df["spread_true"].values, index=pair_df.index))
print("OU spread (true latent series) — AR(1) half-life estimate:")
for k, v in hl_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"  {k}: {v}")

# Theoretical theta=4 -> half-life ln(2)/4 in years if t in years... our OU uses dt=1/252 per step
# Per year theta: half_life_years = ln(2)/theta
print(f"  Model theta (continuous-time param in sim): 4.0 -> t_1/2 in years: {np.log(2)/4:.3f}")


---

## 3. Cointegration & Pairs Trading

### Why quants need this

**Cointegration** formalizes **long-run equilibrium**: two non-stationary series may combine into a **stationary residual**. **Engle–Granger** tests whether OLS residuals are stationary; a significant test supports a **pairs** structure. In production, quants roll estimation, monitor **ADF** on the spread, and **halt** trading when cointegration fails.

### Pairs algorithm (template)

1. Regress $\log P^{(2)} = \alpha + \beta \log P^{(1)} + u$ (in-sample or rolling).
2. Spread $S_t = \log P^{(2)}_t - \hat{\beta} \log P^{(1)}_t$ (demean or include $\hat{\alpha}$ in level).
3. $z_t = (S_t - \text{SMA}_n(S)) / \text{std}_n(S)$.
4. **Enter** short spread (long P1 / short P2) when $z > z_{\text{in}}$; **long spread** when $z < -z_{\text{in}}$; **exit** near 0.
5. Charge **costs** on notional traded.


In [ ]:
def hedge_ratio_ols(log_y: pd.Series, log_x: pd.Series) -> Tuple[float, float]:
    X = sm.add_constant(log_x.values)
    m = OLS(log_y.values, X).fit()
    alpha, beta = float(m.params[0]), float(m.params[1])
    return alpha, beta


def spread_from_betas(log_p1: pd.Series, log_p2: pd.Series, beta: float, alpha: float) -> pd.Series:
    return log_p2 - (alpha + beta * log_p1)


def rolling_zscore(spread: pd.Series, window: int = 60) -> pd.Series:
    m = spread.rolling(window).mean()
    s = spread.rolling(window).std().replace(0, np.nan)
    return (spread - m) / s


# In-sample hedge ratio (first 400 days — illustrate estimation window)
train_end = 400
train = pair_df.iloc[:train_end]
a_hat, b_hat = hedge_ratio_ols(train["log_P2"], train["log_P1"])
# Note: OLS y=log_P2, x=log_P1 -> log_P2 = a + b log_P1; spread = log_P2 - a - b*log_P1
spread_est = spread_from_betas(pair_df["log_P1"], pair_df["log_P2"], b_hat, a_hat)

score, pval, crit = coint(pair_df["log_P2"], pair_df["log_P1"], trend="c")
print(f"Engle-Granger cointegration p-value (full sample): {pval:.4f}")
adf_sp = adfuller(spread_est.dropna(), maxlag=1, autolag=None, regression="c")
print(f"ADF on OLS spread (const): stat={adf_sp[0]:.3f}, p={adf_sp[1]:.4f}")

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(spread_est.index, spread_est, color=COLORS["spread"], lw=1, label="Estimated spread")
ax[0].plot(pair_df.index, pair_df["spread_true"], color=COLORS["neutral"], lw=0.8, alpha=0.5, label="True latent spread")
ax[0].legend(fontsize=8)
ax[0].set_title("Spread: OLS hedge vs. latent OU")
z = rolling_zscore(spread_est, 60)
ax[1].plot(z.index, z, color=COLORS["z"], lw=1)
ax[1].axhline(2, color="gray", ls="--")
ax[1].axhline(-2, color="gray", ls="--")
ax[1].axhline(0, color="black", lw=0.6)
ax[1].set_title("Rolling z-score (60)")
plt.tight_layout()
plt.show()

print(f"Hedge beta (train): {b_hat:.4f}, alpha: {a_hat:.4f}")


In [ ]:
def pairs_backtest(
    p1: pd.Series,
    p2: pd.Series,
    spread: pd.Series,
    z: pd.Series,
    z_in: float = 2.0,
    z_out: float = 0.25,
    cost_bps: float = 5.0,
) -> Tuple[pd.Series, pd.Series]:
    """
    Dollar-neutral unit notionals: position in spread units.
    +1 = long spread (long P2, short P1 proportional to hedge) approximated via returns:
    We use log returns: r_spread ~ r2 - beta*r1 for constant beta (simplified).
    """
    r1 = p1.pct_change()
    r2 = p2.pct_change()
    # Use full-sample beta on log for return approximation
    _, b = hedge_ratio_ols(np.log(p2.loc[spread.dropna().index]), np.log(p1.loc[spread.dropna().index]))
    r_spread = r2 - b * r1

    pos = np.zeros(len(spread))
    state = 0.0
    zv = z.values
    for i in range(1, len(zv)):
        if np.isnan(zv[i]):
            pos[i] = state
            continue
        if state == 0 and zv[i] < -z_in:
            state = 1.0
        elif state == 0 and zv[i] > z_in:
            state = -1.0
        elif state == 1.0 and zv[i] > -z_out:
            state = 0.0
        elif state == -1.0 and zv[i] < z_out:
            state = 0.0
        pos[i] = state

    position = pd.Series(pos, index=spread.index)
    pos_lag = position.shift(1).fillna(0.0)
    strat_ret = pos_lag * r_spread.reindex(spread.index).fillna(0.0)
    turn = position.diff().abs().fillna(0.0) * 2.0
    strat_ret = strat_ret - turn * (cost_bps / 10000.0)
    return strat_ret, position


# Align: use spread from rolling beta would be better; keep simple post-train backtest
spread_full = spread_from_betas(pair_df["log_P1"], pair_df["log_P2"], b_hat, a_hat)
z_full = rolling_zscore(spread_full, 60)
ret_pair, pos_pair = pairs_backtest(pair_df["P1"], pair_df["P2"], spread_full, z_full, 2.0, 0.25, 8.0)
eq = (1 + ret_pair.fillna(0)).cumprod()
print("Pairs trading (z=2 entry, exit near 0, 8 bps round-turn style per unit flip):")
print(f"  Ann. return: {ret_pair.mean()*252:.2%}")
print(f"  Ann. vol:    {ret_pair.std()*np.sqrt(252):.2%}")
print(f"  Sharpe:      {(ret_pair.mean()*252 - 0.03)/(ret_pair.std()*np.sqrt(252)):.2f}" if ret_pair.std() > 0 else "")

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(eq.index, eq, color=COLORS["pnl"], lw=1.5)
ax[0].set_title("Pairs strategy equity (approximate spread returns)")
ax[1].plot(pos_pair.index, pos_pair, drawstyle="steps-post", color=COLORS["spread"])
ax[1].set_title("Position (+1 long spread, -1 short)")
plt.tight_layout()
plt.show()


---

## 4. Bollinger Bands & Z-Scores on Price

### Why quants need this

**Bollinger Bands** plot a moving average plus/minus $k$ standard deviations. A **mean-reversion** heuristic buys when price hits the **lower** band and sells at the **mean** (or opposite for shorts). This is **not** cointegration—it assumes **local stationarity** of price deviations, which is fragile. Quants still use bands as **risk bands**, **volatility gauges**, and **simple baselines**.

### Definition

Middle $M_t = \text{SMA}_n(P_t)$, bands $M_t \pm k \hat{\sigma}_t$ with rolling std $\hat{\sigma}_t$.


In [ ]:
def bollinger(close: pd.Series, n: int = 20, k: float = 2.0) -> pd.DataFrame:
    mid = close.rolling(n).mean()
    sig = close.rolling(n).std()
    return pd.DataFrame({"mid": mid, "upper": mid + k * sig, "lower": mid - k * sig})


def bollinger_mean_reversion(close: pd.Series, n: int = 20, k: float = 2.0, cost_bps: float = 5.0) -> pd.Series:
    bb = bollinger(close, n, k)
    pos = np.zeros(len(close))
    state = 0.0
    c = close.values
    lo = bb["lower"].values
    mi = bb["mid"].values
    hi = bb["upper"].values
    for i in range(n + 1, len(close)):
        if np.isnan(lo[i]):
            pos[i] = state
            continue
        if state == 0 and c[i] < lo[i]:
            state = 1.0
        elif state == 1.0 and c[i] >= mi[i]:
            state = 0.0
        elif state == 0 and c[i] > hi[i]:
            state = -1.0
        elif state == -1.0 and c[i] <= mi[i]:
            state = 0.0
        pos[i] = state
    position = pd.Series(pos, index=close.index)
    r = close.pct_change().fillna(0.0)
    strat = position.shift(1).fillna(0.0) * r
    strat = strat - position.diff().abs().fillna(0.0) * (cost_bps / 10000.0)
    return strat


bb = bollinger(pair_df["P1"], 20, 2.0)
strat_bb = bollinger_mean_reversion(pair_df["P1"], 20, 2.0, 6.0)

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(pair_df.index, pair_df["P1"], label="P1", color=COLORS["a"], lw=1)
ax[0].plot(bb.index, bb["mid"], color=COLORS["neutral"], lw=0.8, label="SMA20")
ax[0].fill_between(bb.index, bb["lower"], bb["upper"], alpha=0.2, color=COLORS["z"])
ax[0].legend(fontsize=8)
ax[0].set_title("Bollinger bands (20, 2σ)")
ax[1].plot(strat_bb.index, (1 + strat_bb.fillna(0)).cumprod(), color=COLORS["pnl"], lw=1)
ax[1].set_title("Bollinger mean-reversion equity (P1)")
plt.tight_layout()
plt.show()


---

## 5. Statistical Arbitrage with Multiple Pairs

### Why quants need this

Single-pair capital is **concentrated** in one relationship. **Multi-pair** books diversify **idiosyncratic** breakdown risk and smooth PnL—at the cost of **correlation** among spreads and **capacity**. A simple construction: run parallel z-score strategies and **weight** spreads by **inverse volatility** or **equal risk**.

### Toy portfolio

We simulate **three** independent cointegrated pairs (different seeds), form spread returns like the single-pair engine, and combine with **equal weights** and **inverse-vol** weights (rolling).


In [ ]:
def multi_pair_returns(seeds: List[int]) -> pd.DataFrame:
    out = {}
    for i, sd in enumerate(seeds):
        df = simulate_cointegrated_pair(seed=sd)
        tr = 400
        a, b = hedge_ratio_ols(df["log_P2"].iloc[:tr], df["log_P1"].iloc[:tr])
        sp = spread_from_betas(df["log_P1"], df["log_P2"], b, a)
        z = rolling_zscore(sp, 60)
        r, _ = pairs_backtest(df["P1"], df["P2"], sp, z, 2.0, 0.25, 5.0)
        out[f"pair_{i}"] = r
    return pd.DataFrame(out).fillna(0.0)


R = multi_pair_returns([42, 43, 44])
vol_roll = R.rolling(60).std().replace(0, np.nan)
w_inv = (1 / vol_roll).div((1 / vol_roll).sum(axis=1), axis=0).fillna(0)
port_eq = R.mean(axis=1)
port_inv = (R * w_inv.shift(1)).sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot((1 + port_eq).cumprod().index, (1 + port_eq).cumprod(), label="Equal weight", lw=1.5)
ax.plot((1 + port_inv).cumprod().index, (1 + port_inv).cumprod(), label="Inv-vol weighted", lw=1.5)
ax.legend()
ax.set_title("Multi-pair stat-arb (toy)")
plt.tight_layout()
plt.show()
print("Ann. Sharpe equal:", (port_eq.mean()*252 - 0.03)/(port_eq.std()*np.sqrt(252)))
print("Ann. Sharpe inv-vol:", (port_inv.mean()*252 - 0.03)/(port_inv.std()*np.sqrt(252)))


---

## 6. Dynamic Position Sizing for Mean Reversion

### Why quants need this

Fixed unit size **ignores signal strength** and **changing spread volatility**. A common approach: scale position with **|z|** (capped) and **inversely** with **realized vol** of spread returns—**vol targeting** on the strategy leg. This mirrors how desks **risk-budget** stat-arb books.

### Rule (illustration)

$$q_t = \text{clip}\left( k \frac{z_t}{\sigma^{\text{spread}}_t}, -q_{\max}, q_{\max} \right)$$
with $\sigma^{\text{spread}}_t$ a rolling volatility of spread returns.


In [ ]:
def pairs_backtest_dynamic(
    p1: pd.Series,
    p2: pd.Series,
    spread: pd.Series,
    z: pd.Series,
    target_ann_vol: float = 0.10,
    z_scale: float = 0.5,
    q_max: float = 3.0,
    cost_bps: float = 5.0,
) -> pd.Series:
    r1 = p1.pct_change()
    r2 = p2.pct_change()
    _, b = hedge_ratio_ols(np.log(p2.loc[spread.dropna().index]), np.log(p1.loc[spread.dropna().index]))
    r_spread = (r2 - b * r1).reindex(spread.index).fillna(0.0)
    sig = r_spread.rolling(20).std().replace(0, np.nan) * np.sqrt(252)
    q = (z_scale * z / sig.replace(0, np.nan)).clip(-q_max, q_max)
    q = q.fillna(0.0)
    scale = (target_ann_vol / sig).replace([np.inf, -np.inf], np.nan).clip(0, 50)
    scale = scale.bfill().fillna(0.0)
    pos = (q * scale).clip(-q_max, q_max)
    pos_lag = pos.shift(1).fillna(0.0)
    strat = pos_lag * r_spread
    turn = pos.diff().abs().fillna(0.0)
    strat = strat - turn * (cost_bps / 10000.0)
    return strat


dyn = pairs_backtest_dynamic(pair_df["P1"], pair_df["P2"], spread_full, z_full)
stat, _ = pairs_backtest(pair_df["P1"], pair_df["P2"], spread_full, z_full, 2.0, 0.25, 8.0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot((1 + stat.fillna(0)).cumprod().index, (1 + stat.fillna(0)).cumprod(), label="Discrete ±1", lw=1.2)
ax.plot((1 + dyn.fillna(0)).cumprod().index, (1 + dyn.fillna(0)).cumprod(), label="Dynamic vol-targeted", lw=1.2)
ax.legend()
ax.set_title("Fixed discrete vs. dynamic sizing (same z, costs)")
plt.tight_layout()
plt.show()

print("\\n" + "=" * 78)
print("MODULE 5.2 — SUMMARY")
print("=" * 78)
print("Stat-arb lives on stationary spreads; test with ADF/coint and monitor breaks.")
print("Half-life from AR(1) guides horizons and window lengths.")
print("Bollinger mean reversion is a different (often weaker) assumption than cointegration.")
print("Multiple pairs diversify idiosyncratic risk; sizing links signal strength to risk.")
print("=" * 78)
